# Train the eye-state detector on a free Colab T4

Trains YOLO26 on the Roboflow open/closed-eyes dataset and gives you `eyes.onnx` for the Raspberry Pi.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*
(you will be asked for your Roboflow key in step 3, or you can upload the dataset zip instead).

Repo: https://github.com/wirepecion/iot-model

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: use Runtime > Change runtime type > T4 GPU, then run again'
print(torch.cuda.get_device_name(0))

## 2. Get the code and install

In [ ]:
!test -d iot-model || git clone -q https://github.com/wirepecion/iot-model.git
%cd iot-model
!pip install -q -U ultralytics roboflow

## 3. Get the dataset (use **one** of the two options)

**Option A: Roboflow API key** (free: roboflow.com → Settings → API Keys). The key is typed into a hidden box and not saved.

**Option B: upload the zip** you downloaded from the dataset page (*Download Dataset → YOLOv11 → download zip to computer*).

In [ ]:
# Option A
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass('Roboflow API key: '))
rf.workspace('yolo-i9xsa').project('sleep-z8rzd').version(6).download('yolov11', location='dataset')

In [ ]:
# Option B (skip if Option A worked)
from google.colab import files

files.upload()                       # choose the dataset .zip
!unzip -q -o *.zip -d dataset

### Check the dataset

In [ ]:
import glob, yaml

cfg = yaml.safe_load(open('dataset/data.yaml'))
print('classes:', cfg['names'])
for split in ('train', 'valid', 'test'):
    print(f'{split:6}', len(glob.glob(f'dataset/{split}/images/*')), 'images')

## 4. Train

The settings below override the defaults in `train/train.py`.
`yolo26n.pt` is the smallest model and the best fit for the Pi. If you want a more accurate one, try `yolo26s.pt`.
If you see *CUDA out of memory*, lower `BATCH` to 8.

In [ ]:
import sys
sys.path.insert(0, 'train')
import train

train.BASE_MODEL = 'yolo26n.pt'
train.IMGSZ = 640
train.BATCH = 16
train.EPOCHS = 100
train.main()                        # trains, prints per-class results on the test images, exports ONNX

## 5. Look at the results

Read the table printed above first: **recall for `close_eyes`** is the number that matters for catching sleep.

In [ ]:
from IPython.display import Image, display

for name in ('results.png', 'confusion_matrix.png'):
    display(Image(f'runs/eyes/{name}', width=800))

### Try the exported ONNX model on a few test images

In [ ]:
import glob
from PIL import Image as PILImage
from ultralytics import YOLO

onnx_model = YOLO('models/eyes.onnx', task='detect')
for r in onnx_model.predict(sorted(glob.glob('dataset/test/images/*'))[:6], imgsz=640, conf=0.35, verbose=False):
    display(PILImage.fromarray(r.plot()[..., ::-1]).resize((320, 320)))

## 6. Download the model

Colab sessions are deleted when you close them, so save the files now.

In [ ]:
from google.colab import files

files.download('models/eyes.onnx')

In [ ]:
# Optional: also keep a copy in your Google Drive
from google.colab import drive

drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/iot-model
!cp models/eyes.onnx runs/eyes/weights/best.pt /content/drive/MyDrive/iot-model/

## Next: use it on the Raspberry Pi

Copy `eyes.onnx` to `models/eyes.onnx` in the repo on the Pi, then follow the *On the Raspberry Pi* section of the README:
`python3 pi/eyes_monitor.py`. Test it on a short clip of your own camera before you trust it.